# LeetCode 3287 — Find the Maximum Sequence Value of Array bằng Python

- Link: https://leetcode.com/problems/find-the-maximum-sequence-value-of-array/
- Độ khó: **Hard**
- Chủ đề: **DP tập giá trị OR đạt được** (giá trị `< 2^7 = 128` nên tập nhỏ), **chia mảng tại một điểm cắt**: `k` phần tử đầu từ tiền tố, `k` phần tử sau từ hậu tố; ghép bằng XOR

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maxValue(self, nums: List[int], k: int) -> int:
```

## 1. Mục tiêu học

1. Dãy con độ dài `2k` = `k` phần tử đầu (nửa trái) + `k` phần tử sau (nửa phải), nửa trái nằm hoàn toàn **trước** nửa phải ⇒ tồn tại điểm cắt `i`: nửa trái ⊆ `nums[:i]`, nửa phải ⊆ `nums[i:]`.
2. `nums[j] < 128` ⇒ mọi OR nằm trong `[0, 127]` ⇒ lưu **tập** các OR đạt được (≤ 128 phần tử) thay vì liệt kê dãy con.
3. DP `L[i]` = tập OR của đúng `k` phần tử chọn từ `nums[:i]`; `R[i]` tương tự cho `nums[i:]`. DP kiểu knapsack theo số phần tử đã chọn.
4. Ghép: `max_{i} max_{a ∈ L[i], b ∈ R[i]} a ⊕ b`, hoặc gọn hơn: chỉ số **sớm nhất** đạt `a` và **muộn nhất** đạt `b`.
5. Kiểm chứng bằng vét cạn tổ hợp.

## 2. Đề bài & ví dụ

Giá trị của dãy `seq` độ dài `2x` là `(seq[0] | … | seq[x−1]) ⊕ (seq[x] | … | seq[2x−1])`. Cho `nums` và `k`, trả về giá trị lớn nhất của một **dãy con** (giữ thứ tự) độ dài `2k`.

**Ví dụ 1:** `[2,6,7]`, `k = 1` → `5` (`[2, 7]`: `2 ⊕ 7`).
**Ví dụ 2:** `[4,2,5,6,7]`, `k = 2` → `2` (`[4,5,6,7]`: `(4|5) ⊕ (6|7) = 5 ⊕ 7`).

**Ràng buộc:** `2 ≤ n ≤ 400`; `1 ≤ nums[i] < 2^7`; `1 ≤ k ≤ n/2`.

In [1]:
from typing import List
from itertools import combinations
from functools import reduce

examples = [([2, 6, 7], 1, 5), ([4, 2, 5, 6, 7], 2, 2)]
for nums, k, exp in examples:
    print(nums, 'k =', k, '-> kỳ vọng', exp)

[2, 6, 7] k = 1 -> kỳ vọng 5
[4, 2, 5, 6, 7] k = 2 -> kỳ vọng 2


## 3. Kiến thức nền

### 3.1. OR bão hòa nhanh — tập OR luôn nhỏ

Mọi OR của các số `< 128` nằm trong `[0, 127]`. Dù có `C(400, 200)` dãy con, tập **giá trị OR** chỉ có tối đa 128 phần tử.

### 3.2. DP knapsack theo số phần tử

`dp[j]` = tập OR của các cách chọn đúng `j` phần tử từ tiền tố đã xét. Thêm `x`: `dp[j] ∪= { v | x : v ∈ dp[j−1] }`, duyệt `j` **giảm dần** để mỗi phần tử dùng tối đa một lần.

In [2]:
def prefix_or_sets(arr, k):
    """out[i] = tập OR của đúng k phần tử chọn từ arr[:i]."""
    dp = [set() for _ in range(k + 1)]
    dp[0] = {0}
    out = [set()]
    for i, x in enumerate(arr):
        for j in range(min(k, i + 1), 0, -1):
            if dp[j - 1]:
                dp[j] |= {v | x for v in dp[j - 1]}
        out.append(set(dp[k]))
    return out

print([sorted(s) for s in prefix_or_sets([4, 2, 5, 6, 7], 2)])

[[], [], [6], [5, 6, 7], [5, 6, 7], [5, 6, 7]]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 2, `k = 2`, cắt tại `i = 3` (`[4, 2, 5] | [6, 7]`):

- `L[3]` = OR của 2 phần tử từ `[4, 2, 5]`: `{4|2=6, 4|5=5, 2|5=7}`.
- `R[3]` = OR của 2 phần tử từ `[6, 7]`: `{7}`.
- `max a ⊕ 7`: `6⊕7 = 1`, `5⊕7 = 2`, `7⊕7 = 0` ⇒ `2` ✓.

Các điểm cắt khác không cho lớn hơn.

## 5. Cách 1 — Brute Force: mọi tổ hợp `2k` chỉ số

In [3]:
def brute(nums, k):
    best = 0
    for idx in combinations(range(len(nums)), 2 * k):
        a = reduce(lambda u, v: u | v, (nums[i] for i in idx[:k]), 0)
        b = reduce(lambda u, v: u | v, (nums[i] for i in idx[k:]), 0)
        best = max(best, a ^ b)
    return best

for nums, k, exp in examples:
    print(brute(nums, k), '| kỳ vọng', exp)

5 | kỳ vọng 5
2 | kỳ vọng 2


**Độ phức tạp:** `O(C(n, 2k) · k)`.

## 6. Ý tưởng thuật toán / bước đột phá

1. `L = prefix_or_sets(nums, k)`, `R' = prefix_or_sets(nums[::-1], k)` ⇒ `R[i] = R'[n − i]` (tập OR của `k` phần tử từ `nums[i:]`).
2. Với mỗi `i ∈ [k, n − k]`: `max_{a ∈ L[i], b ∈ R[i]} a ⊕ b`.

Chi phí DP `O(n · k · 128)` (thực tế nhỏ hơn nhiều vì tập bão hòa), ghép `O(n · 128²)`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DP mảng boolean `128` (cài đặt "chuẩn")

In [4]:
def bool_dp(nums, k):
    n = len(nums)
    def side(arr):
        dp = [[False] * 128 for _ in range(k + 1)]
        dp[0][0] = True
        out = [[False] * 128]
        for i, x in enumerate(arr):
            for j in range(min(k, i + 1), 0, -1):
                prev, cur = dp[j - 1], dp[j]
                for v in range(128):
                    if prev[v]:
                        cur[v | x] = True
            out.append(dp[k][:])
        return out
    L, R = side(nums), side(nums[::-1])
    best = 0
    for i in range(k, n - k + 1):
        A = [a for a in range(128) if L[i][a]]
        B = [b for b in range(128) if R[n - i][b]]
        for a in A:
            for b in B:
                best = max(best, a ^ b)
    return best

for nums, k, exp in examples:
    print(bool_dp(nums, k), '| kỳ vọng', exp)

5 | kỳ vọng 5
2 | kỳ vọng 2


`O(n · k · 128)` ≈ `10^7` phép — chậm trong Python thuần với `n = 400`, nhưng đúng là độ phức tạp đề nhắm tới.

### 7.2. Cách 3 — DP tập (lời giải chính)

In [5]:
def set_dp(nums, k):
    n = len(nums)
    L = prefix_or_sets(nums, k)
    Rr = prefix_or_sets(nums[::-1], k)
    best = 0
    for i in range(k, n - k + 1):
        A, B = L[i], Rr[n - i]
        for a in A:
            for b in B:
                if a ^ b > best:
                    best = a ^ b
    return best

for nums, k, exp in examples:
    print(set_dp(nums, k), '| kỳ vọng', exp)

5 | kỳ vọng 5
2 | kỳ vọng 2


### 7.3. Cách 4 — Chỉ số sớm nhất / muộn nhất cho mỗi giá trị OR

Tập `L[i]` **tăng dần** theo `i` (tiền tố dài hơn không mất lựa chọn). Gọi `first[a]` = `i` nhỏ nhất với `a ∈ L[i]`, `last[b]` = `i` lớn nhất với `b ∈ R[i]`. Cặp `(a, b)` ghép được ⇔ `first[a] ≤ last[b]`. Ghép trong `O(128²)` thay vì `O(n · 128²)`.

In [6]:
def first_last(nums, k):
    n = len(nums)
    L = prefix_or_sets(nums, k)
    Rr = prefix_or_sets(nums[::-1], k)
    INF = n + 1
    first = [INF] * 128
    for i in range(n + 1):
        for a in L[i]:
            if first[a] == INF:
                first[a] = i
    last = [-1] * 128
    for i in range(n + 1):
        for b in Rr[n - i]:
            last[b] = max(last[b], i)
    best = 0
    for a in range(128):
        if first[a] == INF:
            continue
        for b in range(128):
            if last[b] >= first[a] and a ^ b > best:
                best = a ^ b
    return best

for nums, k, exp in examples:
    print(first_last(nums, k), '| kỳ vọng', exp)

5 | kỳ vọng 5
2 | kỳ vọng 2


## 8. TRACE từng bước

Trace cách 3 trên ví dụ 2: in `L[i]`, `R[i]` và cặp tốt nhất ở mỗi điểm cắt.

In [7]:
def trace(nums, k):
    n = len(nums)
    L = prefix_or_sets(nums, k); Rr = prefix_or_sets(nums[::-1], k)
    for i in range(k, n - k + 1):
        A, B = sorted(L[i]), sorted(Rr[n - i])
        a, b = max(((a, b) for a in A for b in B), key=lambda p: p[0] ^ p[1])
        print(f"cắt tại {i}: L = {A}, R = {B} -> tốt nhất {a} ⊕ {b} = {a ^ b}")
    print('đáp án:', set_dp(nums, k), '| vét cạn:', brute(nums, k))

trace([4, 2, 5, 6, 7], 2)

cắt tại 2: L = [6], R = [7] -> tốt nhất 6 ⊕ 7 = 1
cắt tại 3: L = [5, 6, 7], R = [7] -> tốt nhất 5 ⊕ 7 = 2
đáp án: 2 | vét cạn: 2


## 9. Code nộp LeetCode

In [8]:
class Solution:
    def maxValue(self, nums: List[int], k: int) -> int:
        n = len(nums)

        def side(arr):
            dp = [set() for _ in range(k + 1)]
            dp[0] = {0}
            out = [None] * (n + 1)
            out[0] = set()
            for i, x in enumerate(arr):
                for j in range(min(k, i + 1), 0, -1):
                    if dp[j - 1]:
                        dp[j] |= {v | x for v in dp[j - 1]}
                out[i + 1] = dp[k].copy() if i + 1 >= k else set()
            return out

        L = side(nums)
        R = side(nums[::-1])
        first = [n + 1] * 128
        for i in range(k, n + 1):
            for a in L[i]:
                if first[a] > i:
                    first[a] = i
        last = [-1] * 128
        for i in range(k, n + 1):                    # R[i] ứng với nums[n-i:], điểm cắt n - i
            for b in R[i]:
                if last[b] < n - i:
                    last[b] = n - i
        best = 0
        for a in range(128):
            if first[a] > n:
                continue
            fa = first[a]
            for b in range(128):
                if last[b] >= fa and a ^ b > best:
                    best = a ^ b
        return best

for nums, k, exp in examples:
    got = Solution().maxValue(nums, k)
    print(got, 'OK' if got == exp else 'SAI')

5 OK
2 OK


## 10. Vì sao thuật toán đúng

1. **Điểm cắt:** với dãy con chỉ số `i_1 < … < i_{2k}`, chọn cắt `c = i_k + 1` ⇒ nửa trái ⊆ `nums[:c]`, nửa phải ⊆ `nums[c:]`. Ngược lại, mọi cặp (k chỉ số trong `[0, c)`, k chỉ số trong `[c, n)`) tạo một dãy con hợp lệ.
2. **DP tập:** quy nạp theo phần tử — tập OR của `j` phần tử từ tiền tố mới = tập cũ ∪ { OR cũ của `j − 1` phần tử | `x` } (có hoặc không chọn `x`). Duyệt `j` giảm dần tránh dùng `x` hai lần.
3. **Ghép:** giá trị chỉ phụ thuộc cặp OR `(a, b)`; tồn tại điểm cắt chung ⇔ `first[a] ≤ last[b]` vì `L[i]` tăng theo `i` và `R[i]` giảm theo `i`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(C(n, 2k) k)` | `O(k)` |
| 2. DP boolean | `O(n k · 128 + n · 128²)` | `O(n · 128)` |
| 3. DP tập | `O(n k · S + n S²)`, `S ≤ 128` | `O(n S)` |
| 4. Sớm nhất / muộn nhất | `O(n k · S + 128²)` | `O(n S)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [9]:
impls = {'bool_dp': bool_dp, 'set_dp': set_dp, 'first_last': first_last, 'submit': lambda a, k: Solution().maxValue(a, k)}
edge = [([1, 1], 1, 0), ([1, 2], 1, 3), ([127, 1, 1, 127], 2, 0), ([1, 2, 4, 8], 2, 15), ([8, 4, 2, 1], 1, 12)]
for nums, k, exp in examples + edge:
    assert brute(nums, k) == exp, (nums, k, brute(nums, k))
    for name, f in impls.items():
        assert f(nums, k) == exp, (name, nums, k)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [10]:
import random
rng = random.Random(3287)
cnt = 0
for _ in range(400):
    n = rng.randint(2, 10)
    nums = [rng.randint(1, rng.choice([7, 31, 127])) for _ in range(n)]
    k = rng.randint(1, n // 2)
    exp = brute(nums, k)
    for name, f in impls.items():
        assert f(nums, k) == exp, (name, nums, k)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'bộ test.')

Khớp vét cạn trên 400 bộ test.


### 12.3. Ca lớn `n = 400`

In [11]:
import time
big = {
    'ngẫu nhiên 1..127, k = 100': ([rng.randint(1, 127) for _ in range(400)], 100),
    'lũy thừa 2, k = 200': ([1 << rng.randint(0, 6) for _ in range(400)], 200),
    'giá trị nhỏ 1..7, k = 1': ([rng.randint(1, 7) for _ in range(400)], 1),
}
for name, (nums, k) in big.items():
    t = time.time(); a = Solution().maxValue(nums, k); t1 = time.time() - t
    t = time.time(); b = set_dp(nums, k); t2 = time.time() - t
    assert a == b
    print(f"{name:28s}: {a:4d} | nộp {t1:.2f}s | cách 3 {t2:.2f}s")
nums = [1] * 200 + [126] * 200
assert Solution().maxValue(nums, 200) == 1 ^ 126
print('[1]*200 + [126]*200, k = 200: đáp án 127 — đúng.')

ngẫu nhiên 1..127, k = 100  :   96 | nộp 0.13s | cách 3 0.11s


lũy thừa 2, k = 200         :    0 | nộp 0.44s | cách 3 0.40s
giá trị nhỏ 1..7, k = 1     :    7 | nộp 0.00s | cách 3 0.00s
[1]*200 + [126]*200, k = 200: đáp án 127 — đúng.


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chọn **đoạn con liên tiếp** thay vì dãy con

In [12]:
def wrong_subarray(nums, k):
    best = 0
    for s in range(len(nums) - 2 * k + 1):
        a = reduce(lambda u, v: u | v, nums[s:s + k], 0)
        b = reduce(lambda u, v: u | v, nums[s + k:s + 2 * k], 0)
        best = max(best, a ^ b)
    return best

print('sai :', wrong_subarray(*examples[0][:2]), '| đúng:', Solution().maxValue(*examples[0][:2]))

sai : 4 | đúng: 5


**Vì sao sai:** dãy con được bỏ phần tử ở giữa (`[2, 7]` bỏ qua `6`).

### Lỗi 2 — Chỉ lấy OR **lớn nhất** mỗi phía

In [13]:
def wrong_max_or(nums, k):
    n = len(nums)
    L = prefix_or_sets(nums, k); Rr = prefix_or_sets(nums[::-1], k)
    best = 0
    for i in range(k, n - k + 1):
        best = max(best, max(L[i]) ^ max(Rr[n - i]))             # SAI: XOR không đơn điệu theo OR
    return best

print('sai :', wrong_max_or(*examples[1][:2]), '| đúng:', Solution().maxValue(*examples[1][:2]))

sai : 1 | đúng: 2


**Vì sao sai:** XOR lớn cần hai OR **khác nhau** ở bit cao; OR lớn nhất hai phía có thể trùng nhau (`7 ⊕ 7 = 0`).

### Lỗi 3 — Cho phép chọn **tối đa** `k` phần tử mỗi phía

In [14]:
def wrong_at_most_k(nums, k):
    n = len(nums)
    def side(arr):
        dp = [set() for _ in range(k + 1)]; dp[0] = {0}; out = [set()]
        for i, x in enumerate(arr):
            for j in range(min(k, i + 1), 0, -1):
                if dp[j - 1]:
                    dp[j] |= {v | x for v in dp[j - 1]}
            out.append(set().union(*dp[1:]))                    # SAI: gộp mọi j <= k
        return out
    L, Rr = side(nums), side(nums[::-1])
    best = 0
    for i in range(1, n):
        for a in L[i]:
            for b in Rr[n - i]:
                best = max(best, a ^ b)
    return best

print('sai :', wrong_at_most_k(*examples[1][:2]), '| đúng:', Solution().maxValue(*examples[1][:2]))

sai : 7 | đúng: 2


**Vì sao sai:** đề yêu cầu dãy con độ dài **đúng** `2k`; chọn ít phần tử hơn cho OR nhỏ hơn và có thể tạo XOR lớn hơn đáp án thật (ví dụ `4 ⊕ 7 = 3` với một phần tử mỗi phía).

### Lỗi 4 — Duyệt `j` **tăng dần** trong knapsack (dùng một phần tử nhiều lần)

In [15]:
def wrong_increasing_j(nums, k):
    n = len(nums)
    def side(arr):
        dp = [set() for _ in range(k + 1)]; dp[0] = {0}; out = [set()]
        for x in arr:
            for j in range(1, k + 1):                           # SAI: dp[j-1] có thể đã chứa x
                if dp[j - 1]:
                    dp[j] |= {v | x for v in dp[j - 1]}
            out.append(set(dp[k]))
        return out
    L, Rr = side(nums), side(nums[::-1])
    best = 0
    for i in range(0, n + 1):
        for a in L[i]:
            for b in Rr[n - i]:
                best = max(best, a ^ b)
    return best

found = None
r_ = random.Random(5)
for _ in range(20000):
    n_ = r_.randint(2, 7); a_ = [r_.randint(1, 15) for _ in range(n_)]; k_ = r_.randint(1, n_ // 2)
    if wrong_increasing_j(a_, k_) != brute(a_, k_):
        if found is None or len(a_) < len(found[0]):
            found = (a_, k_)
print('phản ví dụ:', found, '-> sai:', wrong_increasing_j(*found), '| đúng:', Solution().maxValue(*found))

phản ví dụ: ([9, 6, 15, 9], 2) -> sai: 15 | đúng: 0


**Vì sao sai:** duyệt `j` tăng dần thì sau khi cập nhật `dp[1]` bằng `x`, `dp[2]` lại dùng `dp[1]` vừa chứa `x` ⇒ như chọn `x` hai lần; "đủ `k` phần tử" sớm hơn thực tế.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Giá trị bị chặn nhỏ (`< 128`, `< 1024`) với phép OR/AND ⇒ DP trên **tập giá trị đạt được**.
- "Dãy con chia làm hai nửa có thứ tự" ⇒ duyệt điểm cắt, DP tiền tố + DP hậu tố.
- XOR không đơn điệu ⇒ phải giữ **toàn bộ** tập, không tham lam theo giá trị lớn nhất.

## 15. Biến thể của bài

1. Thay XOR bằng `a & b`, `a | b` hay `a − b`.
2. Hai nửa có độ dài khác nhau `p`, `q`.
3. Hai nửa **không** cần theo thứ tự (chỉ cần rời nhau) ⇒ DP 2 chiều số phần tử mỗi nhóm.
4. Giá trị lên tới `2^{10}` ⇒ tập tới 1024 — vẫn dùng được với `n` nhỏ.

In [16]:
# Biến thể 2: độ dài p và q
def unequal(nums, p, q):
    n = len(nums)
    L = prefix_or_sets(nums, p); Rr = prefix_or_sets(nums[::-1], q)
    best = -1
    for i in range(p, n - q + 1):
        for a in L[i]:
            for b in Rr[n - i]:
                best = max(best, a ^ b)
    return best

rng = random.Random(15)
for _ in range(200):
    n = rng.randint(2, 9); nums = [rng.randint(1, 31) for _ in range(n)]
    p = rng.randint(1, n - 1); q = rng.randint(1, n - p)
    exp = max(reduce(lambda u, v: u | v, (nums[i] for i in c[:p]), 0) ^ reduce(lambda u, v: u | v, (nums[i] for i in c[p:]), 0)
              for c in combinations(range(n), p + q))
    assert unequal(nums, p, q) == exp
print('Biến thể độ dài p, q khớp vét cạn.')

Biến thể độ dài p, q khớp vét cạn.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Mọi tổ hợp | mũ | Oracle |
| 2 | DP boolean 128 | `O(128 n k)` | Chuẩn, chậm trong Python |
| 3 | DP tập | `O(n k S)` | Nhanh do tập bão hòa |
| 4 | Sớm nhất / muộn nhất | `+ O(128²)` | **Nộp bài** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `[3, 3, 3, 3]`, `k = 2`.

**Bài 2 (tính tay):** `[1, 2, 3]`, `k = 1`.

**Bài 3 (điền TODO):** trả về luôn **dãy con** đạt giá trị lớn nhất.

**Bài 4 (tự viết lại):** thay XOR bằng AND.

**Bài 5 (mini challenge):** biến thể 3 — hai nhóm rời nhau, không cần thứ tự.

In [17]:
def witness_todo(nums, k):
    # TODO
    raise NotImplementedError

def max_and_todo(nums, k):
    # TODO
    raise NotImplementedError

def unordered_groups_todo(nums, k):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Hai phía đều OR = 3. (2) `3 ⊕ 3`. (3) `0`.

**Bài 2.** (1) Cặp `(1,2)`, `(1,3)`, `(2,3)`. (2) XOR `3, 2, 1`. (3) `3`.

**Bài 3.** (1) DP tập kèm "truy vết": với mỗi `(j, v)` lưu chỉ số phần tử cuối và giá trị trước. (2) Tìm `(a, b, i)` tối ưu. (3) Dựng ngược hai phía.

**Bài 4.** (1) Giống hệt, chỉ thay phép ghép. (2) `max a & b`. (3) Vét cạn.

**Bài 5.** (1) DP `(j1, j2, or1, or2)` quá lớn? `or1, or2 < 128`, `j ≤ k` ⇒ dùng tập cặp. (2) Mỗi phần tử: bỏ / nhóm 1 / nhóm 2. (3) Chỉ dùng cho `n` nhỏ.

## 19. Đáp án tham khảo cho bài tập

In [18]:
# Bài 1, 2, 4
assert Solution().maxValue([3, 3, 3, 3], 2) == 0
assert Solution().maxValue([1, 2, 3], 1) == 3

def max_and(nums, k):
    n = len(nums)
    L = prefix_or_sets(nums, k); Rr = prefix_or_sets(nums[::-1], k)
    return max(a & b for i in range(k, n - k + 1) for a in L[i] for b in Rr[n - i])

rng = random.Random(3)
for _ in range(200):
    n = rng.randint(2, 9); nums = [rng.randint(1, 31) for _ in range(n)]; k = rng.randint(1, n // 2)
    exp = max(reduce(lambda u, v: u | v, (nums[i] for i in c[:k]), 0) & reduce(lambda u, v: u | v, (nums[i] for i in c[k:]), 0)
              for c in combinations(range(n), 2 * k))
    assert max_and(nums, k) == exp
print('Bài 1: 0 | Bài 2: 3 | Bài 4 (AND) khớp vét cạn.')

Bài 1: 0 | Bài 2: 3 | Bài 4 (AND) khớp vét cạn.


In [19]:
# Bài 3, 5
def or_sets_with_trace(arr, k):
    """trace[i][(j, v)] = (chỉ số cuối, v trước) cho một cách đạt v với j phần tử trong arr[:i]."""
    dp = {(0, 0): None}
    snapshots = [dict(dp)]
    for i, x in enumerate(arr):
        new = dict(dp)
        for (j, v), _ in dp.items():
            if j < k and (j + 1, v | x) not in new:
                new[(j + 1, v | x)] = (i, j, v)
        dp = new
        snapshots.append(dict(dp))
    return snapshots

def rebuild(snap, j, v):
    idx = []
    while j:
        i, pj, pv = snap[(j, v)]
        idx.append(i); j, v = pj, pv
    return idx[::-1]

def witness(nums, k):
    n = len(nums)
    L = or_sets_with_trace(nums, k); R = or_sets_with_trace(nums[::-1], k)
    best = (-1, None)
    for i in range(k, n - k + 1):
        for (j, a) in L[i]:
            if j != k:
                continue
            for (j2, b) in R[n - i]:
                if j2 == k and a ^ b > best[0]:
                    best = (a ^ b, (i, a, b))
    val, (i, a, b) = best
    left = rebuild(L[i], k, a)
    right = sorted(n - 1 - t for t in rebuild(R[n - i], k, b))
    return val, [nums[t] for t in left + right]

def unordered_groups(nums, k):
    states = {(0, 0, 0, 0)}
    for x in nums:
        new = set(states)
        for j1, j2, o1, o2 in states:
            if j1 < k:
                new.add((j1 + 1, j2, o1 | x, o2))
            if j2 < k:
                new.add((j1, j2 + 1, o1, o2 | x))
        states = new
    return max(o1 ^ o2 for j1, j2, o1, o2 in states if j1 == j2 == k)

rng = random.Random(4)
for _ in range(150):
    n = rng.randint(2, 8); nums = [rng.randint(1, 31) for _ in range(n)]; k = rng.randint(1, n // 2)
    val, seq = witness(nums, k)
    assert val == brute(nums, k) and len(seq) == 2 * k
    assert reduce(lambda u, v: u | v, seq[:k]) ^ reduce(lambda u, v: u | v, seq[k:]) == val
    exp5 = 0
    for c in combinations(range(n), 2 * k):
        for g in combinations(c, k):
            rest = [t for t in c if t not in g]
            exp5 = max(exp5, reduce(lambda u, v: u | v, (nums[t] for t in g)) ^ reduce(lambda u, v: u | v, (nums[t] for t in rest)))
    assert unordered_groups(nums, k) == exp5
print('Bài 3 (dựng dãy con) và Bài 5 (không thứ tự) khớp vét cạn. Ví dụ:', witness(*examples[1][:2]))

Bài 3 (dựng dãy con) và Bài 5 (không thứ tự) khớp vét cạn. Ví dụ: (2, [4, 5, 6, 7])


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
side(arr): dp[0] = {0}; với mỗi x: for j = k..1: dp[j] |= {v | x for v in dp[j-1]}; ghi dp[k] theo tiền tố
L = side(nums); R = side(nums[::-1])
first[a] = điểm cắt sớm nhất có a ∈ L; last[b] = điểm cắt muộn nhất có b ∈ R
ans = max a ^ b với first[a] <= last[b]
```

**Checklist:**

- [ ] Dãy con (không phải đoạn con)?
- [ ] Đúng `k` phần tử mỗi phía?
- [ ] Knapsack duyệt `j` giảm dần?
- [ ] Giữ toàn bộ tập OR (không tham lam max)?
- [ ] Điểm cắt trong `[k, n − k]`?